# SocksLaundering

[Codility task](https://app.codility.com/programmers/trainings/3/socks_laundering/) - Exercise 3, 2017 Contest

Bob has clean socks `C` and dirty socks `D` (colours are integers). He can wash at most `K` dirty socks. `solution(K, C, D)` returns the maximum number of same-colour pairs of clean socks he can have afterwards.

Example: `K = 2`, `C = [1, 2, 1, 1]`, `D = [1, 4, 3, 2, 4]` -> 3 (wash one sock of colour 1 and one of colour 2).

Assumptions: `K` in `[0..50]`, colours in `[1..50]`, `C` and `D` have at most 50 elements.

## Idea

Work colour by colour. A colour with `n` clean socks already forms `n // 2` pairs.

1. If `n` is **odd**, one clean sock is alone. Washing **one** dirty sock of that colour creates a pair: cost 1 sock, gain 1 pair.
2. Any other gain needs **two** dirty socks of the same colour: cost 2 socks, gain 1 pair.

Cost-1 gains are strictly better than cost-2 gains, so be greedy:

1. spend up to `K` washes on the single socks of odd colours that have a dirty sock;
2. with the remaining capacity wash dirty pairs: `min(remaining // 2, number of dirty pairs)`. For a colour whose single sock was already used, only `dirty - 1` socks remain to form pairs.

Time O(N + M).

In [ ]:
from collections import Counter


def solution(K, C, D):
    clean = Counter(C)
    dirty = Counter(D)

    pairs = sum(n // 2 for n in clean.values())                   # pairs already in the clean drawer

    singles = {color for color, n in clean.items() if n % 2 == 1 and dirty[color] > 0}
    used = min(K, len(singles))                                    # one wash -> one new pair
    pairs += used
    K -= used

    available = 0                                                  # pairs that can still be taken from the dirty drawer
    for color, n in dirty.items():
        if color in singles:
            n -= 1                                                 # its first dirty sock completed the clean single
        available += n // 2
    return pairs + min(K // 2, available)                          # two washes -> one new pair

## Official example

In [ ]:
assert solution(2, [1, 2, 1, 1], [1, 4, 3, 2, 4]) == 3
print("example OK")

## Edge cases

In [ ]:
assert solution(0, [1, 1, 2], [2, 2]) == 1                   # K = 0: only the clean pairs count
assert solution(0, [1], [1]) == 0
assert solution(1, [1], [1]) == 1                            # one wash completes the pair
assert solution(1, [1], [2, 2]) == 0                         # one wash cannot make a pair of colour 2
assert solution(2, [1], [2, 2]) == 1                         # two washes make a dirty pair
assert solution(3, [1], [1, 2, 2]) == 2                      # single (1 sock) + dirty pair (2 socks)
assert solution(50, [1, 2, 3], [4, 5, 6]) == 0               # nothing can be paired
assert solution(5, [1, 1, 1], [1, 1, 1, 1]) == 3             # the single (1 wash) + one dirty pair (2 washes); one wash is left over
assert solution(1, [1, 1, 1], [1, 1, 1, 1]) == 2             # K = 1: the single sock is the best use
assert solution(3, [1, 2], [1, 2, 3, 3]) == 2                # two singles (2 washes); remaining 1 cannot pair
assert solution(4, [1, 2], [1, 2, 3, 3]) == 3                # two singles + the dirty pair of colour 3
print("edge cases OK")

## Brute force

The reference tries **every subset** of at most `K` dirty socks, counts the pairs of the resulting clean drawer and keeps the best. It is exponential, so only small arrays are used (up to 9 dirty socks, 4 colours).

In [ ]:
import itertools
import random


def brute(K, C, D):
    best = 0
    for size in range(0, min(K, len(D)) + 1):
        for chosen in itertools.combinations(range(len(D)), size):
            socks = C + [D[i] for i in chosen]
            best = max(best, sum(n // 2 for n in Counter(socks).values()))
    return best


rng = random.Random(7)
for _ in range(4000):
    C = [rng.randint(1, 4) for _ in range(rng.randint(1, 8))]
    D = [rng.randint(1, 4) for _ in range(rng.randint(1, 9))]
    K = rng.randint(0, 9)
    assert solution(K, C, D) == brute(K, C, D), (K, C, D)
print("matches brute force on 4000 random cases")

## Performance (the limits are tiny, this just confirms it)

In [ ]:
import time

C = [rng.randint(1, 50) for _ in range(50)]
D = [rng.randint(1, 50) for _ in range(50)]
start = time.perf_counter()
for _ in range(10_000):
    solution(50, C, D)
print(f"10,000 calls at the maximum size: {time.perf_counter() - start:.3f}s")

## Notes

- The tricky part is the greedy order: single socks first (cost 1), pairs of dirty socks second (cost 2).
- When a colour is odd in the clean drawer, its first dirty sock is "used up" by the single; only the others can form pairs.
- If the capacity left is odd, one wash is simply wasted: a lone dirty sock does not create a pair.